In [ ]:
!pip -q install langgraph langchain langchain-groq


# =========================
# IMPORTS
# =========================
import os                  # for environment variables (API keys)
import re                  # for extracting numbers from text
from typing import TypedDict, Optional, Dict, Any, List  # type safety

from langgraph.graph import StateGraph   # core LangGraph class
from langchain_groq import ChatGroq      # LLM provider
from google.colab import userdata        # secure secrets in Colab


# =========================
# LOAD API KEY SECURELY
# =========================
# Fetch API key from Colab Secrets (not hardcoded → secure)
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")


# =========================
# INITIALIZE LLM
# =========================
# This creates a connection to the LLM model
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",   # powerful model
    groq_api_key=os.environ["GROQ_API_KEY"]
)

### helper functiom -> simplify LLm calling

def ask_llm(prompt: str) -> str:
  # send prompt to model and return clean output
    return llm.invoke(prompt).content.strip()

## state definition
### this defines what data flows between nodes( agents) in langgraph

class BotState(TypedDict, total=False):
  user_input: str         ## what user typed
  intent: Optional[str]       ### detect intent
  data: Optional[str]        ### response shown to user
  expense: List[Dict[str, Any]]        ### list of expense
  hitl_flag: bool          #### risk flag ( humain in the loop)


## define risky financial behaviour

def is_high_risk(text: str) -> bool:
  risky_keywords = [
      "all-in", "sell my house", "crypto all", "bet everything", "withdraw my entire savings", "theft", "cancel"
  ]
  return any ( k in text for k in risky_keywords)


## extract number from user input

def parse_amount(text: str):
  ## find numbers

  match = re.search(r"\d+", text)
  return float(match.group()) if match else None


## creating nodes ( core logic )
### 1st node detecting intent

def node_intent(state: BotState) -> BotState:
  text = state.get("user_input","")

  ## check if user input is something risky


  state["hitl_flag"] = is_high_risk(text.lower())

  ## send to LLM for classification

  prompt = f"""
  Classify into one word : expense, budget, advice, unknown.

  Examples:

  "Add 200" -> Expense
  "Total Spend?" -> budget
  "How to save?" -> advice

  User: "{text}"

  Answer:
  """

  ## invoke LLM to get prediction on intent

  intent = ask_llm(prompt).lower().strip()

  ## safety check for unknown

  if intent not in ["expense", "budget", "advice"]:
    intent = "unknown"

  state["intent"] = intent

  return state

### 2nd node expense node

def node_expense(state: BotState) -> BotState:
  amt = parse_amount(state.get("user_input", ""))

  ## if no amount is found
  if amt is None:
    state["data"] = "please enter any amount ( e.g Add 1000)"
    return state


  ## add expense
  state.setdefault("expense",[]).append({"amount": amt})

  ## return to user
  state["data"] = f"added {amt} to your expense"
  return state

### 3rd node ... budget node

def node_budget(state: BotState) -> BotState:
  exps = state.get("expense", [])
  ### if no expense -> inform user
  if not exps:
    state["data"] = "please add some expenses first"
    return state

  ## sum all the expense
  total = sum(exp["amount"] for exp in exps)

  ## return to user
  state["data"] = f"your total expense is {total}"
  return state

### 4th node advice ( LLM will recommend you)

def node_advice(state: BotState) -> BotState:
  prompt = f"""
  Give some money saving tips.

  User: "{state["user_input"]}"

  """

### generate some helpfull tipls using LLM
  state["data"] = ask_llm(prompt)
  return state

### 5th node risk node

def node_hitl(state: BotState) -> BotState:
  state["data"] = " High risk financial transaction detetcted"
  return state



## 6th fall back node

def node_fallback(state: BotState) -> BotState:
  state["data"] = "I'm sorry, I didn't understand that."
  return state

########## end of nodes logic#############################


### decision logic

def choose_next(state: BotState) -> str:

  if state.get("hitl_flag"):
    return "hitl"

  return state.get("intent", "fallback")

### langgraph workflow

builder = StateGraph(BotState)

## add nodes ( steps in workflow)

builder.add_node("intent", node_intent)
builder.add_node("expense", node_expense)
builder.add_node("budget", node_budget)
builder.add_node("advice", node_advice)
builder.add_node("hitl", node_hitl)
builder.add_node("fallback", node_fallback)

### define starting point

builder.set_entry_point("intent")

### define routing logic

builder.add_conditional_edges(
    "intent",
    choose_next,
    {
        "expense": "expense",
        "budget": "budget",
        "advice": "advice",
        "hitl": "hitl",
        "fallback": "fallback",
        "unknown": "fallback"
    },
)


### compile it

graph = builder.compile()


#### run chat loop

def run_chat():
  print(" Simple finance bot\n")

  ### initial state( memory)

  state: BotState = {
      "expense": [], # Changed to expense (singular) to match TypedDict
      "hitl_flag": False # Changed false to False

  }

  while True:
    msg = input("You define your financial scenario: ")


    ## exit condition

    if msg.lower() in ("exit", "quit"):
      print("Bot Bye: ")
      break

    ### state state with user input

    state["user_input"] = msg


    ## run langraph
    state = graph.invoke(state)

    ## print response
    print("Bot:", state["data"])
    print()


### start chatbot
run_chat()



 Simple finance bot

You define your financial scenario: sell my house
Bot:  High risk financial transaction detetcted

You define your financial scenario: expense 500
Bot: added 500.0 to your expense

You define your financial scenario: travel 200
Bot: added 200.0 to your expense

You define your financial scenario: total expense
Bot: your total expense is 700.0

You define your financial scenario: earning 1000
Bot: your total expense is 700.0

You define your financial scenario: what is my budget
Bot: your total expense is 700.0

You define your financial scenario: give some financial tips
Bot: Here are some valuable financial tips to help you save money:

1. **Create a budget**: Track your income and expenses to understand where your money is going. Make a budget that accounts for all necessary expenses, savings, and debt repayment.
2. **Automate your savings**: Set up automatic transfers from your checking account to your savings or investment accounts. This way, you'll ensure that